In [1]:
# ============================================================
# NB05_Silver_Fundamental_Features (MERGED: Old + 4-Year)
# ============================================================

from pyspark.sql import Window
from pyspark.sql import functions as F
from datetime import datetime
import uuid

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

print(f"RUN_ID: {RUN_ID}")
print(f"RUN_TIMESTAMP: {RUN_TIMESTAMP}")

# ------------------------------------------------------------
# 1. Load Bronze Inputs
# ------------------------------------------------------------
fund_long = spark.read.table("bronze_fundamentals_long")
market_snap = spark.read.table("bronze_market_snapshot").dropDuplicates(["symbol"])
universe = spark.read.table("bronze_universe_snapshot")

eligible = (
    universe
    .filter((F.col("instrument_type") == "EQUITY") & (F.col("eligible_for_stock_screen") == True))
    .select("symbol")
    .dropDuplicates(["symbol"])
)

print(f"Eligible stocks: {eligible.count()}")

# ------------------------------------------------------------
# 2. Latest Annual Snapshot (for scoring — same as old)
# ------------------------------------------------------------
w_ann_metric = Window.partitionBy("symbol", "metric_name").orderBy(F.to_date("period_end").desc())

ann_latest = (
    fund_long
    .filter(F.col("period_type") == "Annual")
    .withColumn("period_end_dt", F.to_date("period_end"))
    .withColumn("rn", F.row_number().over(w_ann_metric))
    .filter(F.col("rn") == 1)
)

ann_pivot = (
    ann_latest
    .groupBy("symbol")
    .pivot("metric_name", [
        "EBIT", "Total Revenue", "Net Income", "Diluted EPS", "Basic EPS",
        "Total Current Assets", "Total Current Liabilities",
        "Total Debt", "Long Term Debt",
        "Cash And Cash Equivalents", "Net PPE",
        "Total Equity", "Stockholders Equity", "Total Shareholder Equity",
        "Operating Cash Flow", "Capital Expenditure"
    ])
    .agg(F.first("metric_value"))
)

ann_pivot = (
    ann_pivot
    .withColumn(
        "Total_Equity_Clean",
        F.coalesce("Total Equity", "Stockholders Equity", "Total Shareholder Equity")
    )
    .drop("Total Equity", "Stockholders Equity", "Total Shareholder Equity")
    .withColumnRenamed("Total_Equity_Clean", "Total_Equity")
)

# Latest annual date per symbol
w_ann_date = Window.partitionBy("symbol").orderBy(F.to_date("period_end").desc())
ann_dates = (
    fund_long
    .filter(F.col("period_type") == "Annual")
    .withColumn("rn", F.row_number().over(w_ann_date))
    .filter(F.col("rn") == 1)
    .select("symbol", F.to_date("period_end").alias("period_end_dt"))
)
ann_pivot = ann_pivot.join(ann_dates, on="symbol", how="left")

# ------------------------------------------------------------
# 3. Last 4 Annual EPS (for 3-year CAGR — same as old)
# ------------------------------------------------------------
w_ann_eps = Window.partitionBy("symbol").orderBy(F.to_date("period_end").desc())

ann_eps_pivot = (
    fund_long
    .filter((F.col("period_type") == "Annual") & (F.col("metric_name") == "Diluted EPS"))
    .withColumn("period_end_dt", F.to_date("period_end"))
    .withColumn("eps_rn", F.row_number().over(w_ann_eps))
    .filter(F.col("eps_rn") <= 4)
    .groupBy("symbol")
    .pivot("eps_rn", [1, 2, 3, 4])
    .agg(F.first("metric_value"))
    .withColumnRenamed("1", "Diluted_EPS_ann")
    .withColumnRenamed("2", "Diluted_EPS_ann_lag1")
    .withColumnRenamed("3", "Diluted_EPS_ann_lag2")
    .withColumnRenamed("4", "Diluted_EPS_ann_lag3")
)

# ------------------------------------------------------------
# 4. Latest Quarterly (for scoring — same as old)
# ------------------------------------------------------------
w_qtr_metric = Window.partitionBy("symbol", "metric_name").orderBy(F.to_date("period_end").desc())

qtr_latest = (
    fund_long
    .filter(F.col("period_type") == "Quarterly")
    .withColumn("period_end_dt_q", F.to_date("period_end"))
    .withColumn("rn_q", F.row_number().over(w_qtr_metric))
    .filter(F.col("rn_q") == 1)
)

qtr_pivot_latest = (
    qtr_latest
    .groupBy("symbol")
    .pivot("metric_name", ["Diluted EPS", "Total Revenue"])
    .agg(F.first("metric_value"))
    .withColumnRenamed("Diluted EPS", "Diluted_EPS_qtr")
    .withColumnRenamed("Total Revenue", "Total_Revenue_qtr")
)

w_qtr_date = Window.partitionBy("symbol").orderBy(F.to_date("period_end").desc())
qtr_dates = (
    fund_long
    .filter(F.col("period_type") == "Quarterly")
    .withColumn("rn", F.row_number().over(w_qtr_date))
    .filter(F.col("rn") == 1)
    .select("symbol", F.to_date("period_end").alias("period_end_dt_q"))
)
qtr_pivot_latest = qtr_pivot_latest.join(qtr_dates, on="symbol", how="left")

# Quarterly lag 4 (same quarter last year — same as old)
qtr_ranked = (
    fund_long
    .filter(F.col("period_type") == "Quarterly")
    .filter(F.col("metric_name").isin("Diluted EPS", "Total Revenue"))
    .withColumn("period_end_dt_q", F.to_date("period_end"))
    .withColumn("q_rn", F.row_number().over(w_qtr_metric))
    .filter(F.col("q_rn") <= 4)
)

qtr_lag4 = (
    qtr_ranked
    .filter(F.col("q_rn") == 4)
    .groupBy("symbol")
    .pivot("metric_name", ["Diluted EPS", "Total Revenue"])
    .agg(F.first("metric_value"))
    .withColumnRenamed("Diluted EPS", "Diluted_EPS_qtr_lag4")
    .withColumnRenamed("Total Revenue", "Total_Revenue_qtr_lag4")
)

# ------------------------------------------------------------
# 5. 4-Year Annual Metrics
# ------------------------------------------------------------
w_ann_hist = Window.partitionBy("symbol", "metric_name").orderBy(F.to_date("period_end").desc())

ann_4y_agg = (
    fund_long
    .filter(F.col("period_type") == "Annual")
    .withColumn("yr_rn", F.dense_rank().over(w_ann_hist))
    .filter(F.col("yr_rn") <= 4)
    .groupBy("symbol")
    .agg(
        F.sum(F.when((F.col("metric_name") == "Net Income") & (F.col("metric_value") > 0), 1).otherwise(0)).alias("positive_earnings_4y_count"),
        F.sum(F.when(F.col("metric_name") == "Operating Cash Flow", F.col("metric_value")).otherwise(0)).alias("cum_4yr_ocf"),
        F.sum(F.when(F.col("metric_name") == "Net Income", F.col("metric_value")).otherwise(0)).alias("cum_4yr_ni"),
        F.max(F.when((F.col("metric_name") == "Diluted EPS") & (F.col("yr_rn") == 1), F.col("metric_value"))).alias("eps_y0"),
        F.max(F.when((F.col("metric_name") == "Diluted EPS") & (F.col("yr_rn") == 2), F.col("metric_value"))).alias("eps_y1"),
        F.max(F.when((F.col("metric_name") == "Diluted EPS") & (F.col("yr_rn") == 3), F.col("metric_value"))).alias("eps_y2"),
        F.max(F.when((F.col("metric_name") == "Diluted EPS") & (F.col("yr_rn") == 4), F.col("metric_value"))).alias("eps_y3"),
        F.max(F.when((F.col("metric_name") == "Total Debt") & (F.col("yr_rn") == 1), F.col("metric_value"))).alias("debt_y0"),
        F.max(F.when((F.col("metric_name") == "Total Debt") & (F.col("yr_rn") == 4), F.col("metric_value"))).alias("debt_y3"),
        F.max(F.when((F.col("metric_name") == "Total Revenue") & (F.col("yr_rn") == 1), F.col("metric_value"))).alias("rev_y0"),
        F.max(F.when((F.col("metric_name") == "Total Revenue") & (F.col("yr_rn") == 4), F.col("metric_value"))).alias("rev_y3"),
        F.max(F.when((F.col("metric_name") == "Operating Cash Flow") & (F.col("yr_rn") == 1), F.col("metric_value"))).alias("ocf_y0"),
        F.max(F.when((F.col("metric_name") == "Operating Cash Flow") & (F.col("yr_rn") == 4), F.col("metric_value"))).alias("ocf_y3")
    )
    .withColumn(
        "eps_cagr_4y",
        F.when(
            (F.col("eps_y3") > 0) & (F.col("eps_y0") > 0),
            F.pow(F.col("eps_y0") / F.col("eps_y3"), 1.0/3.0) - 1.0
        ).otherwise(None)
    )
    .withColumn(
        "rev_cagr_4y",
        F.when(
            (F.col("rev_y3") > 0) & (F.col("rev_y0") > 0),
            F.pow(F.col("rev_y0") / F.col("rev_y3"), 1.0/3.0) - 1.0
        ).otherwise(None)
    )
    .withColumn(
        "ocf_cagr_4y",
        F.when(
            (F.col("ocf_y3") > 0) & (F.col("ocf_y0") > 0),
            F.pow(F.col("ocf_y0") / F.col("ocf_y3"), 1.0/3.0) - 1.0
        ).otherwise(None)
    )
    .withColumn(
        "cum_4yr_ocf_to_ni",
        F.when(F.col("cum_4yr_ni") > 0, F.col("cum_4yr_ocf") / F.col("cum_4yr_ni")).otherwise(None)
    )
    .withColumn(
        "is_deleveraging",
        F.when(
            (F.col("debt_y0").isNotNull()) & (F.col("debt_y3").isNotNull()),
            F.col("debt_y0") <= F.col("debt_y3")
        ).otherwise(None)
    )
)

# ------------------------------------------------------------
# 6. Quarterly EPS YoY (only q_rn 1 and 5, i.e. same quarter last year)
# ------------------------------------------------------------
w_qtr_eps = Window.partitionBy("symbol").orderBy(F.to_date("period_end").desc())

qtr_eps_yoy = (
    fund_long
    .filter((F.col("period_type") == "Quarterly") & (F.col("metric_name") == "Diluted EPS"))
    .withColumn("q_rn", F.row_number().over(w_qtr_eps))
    .filter(F.col("q_rn").isin(1, 5))
    .groupBy("symbol")
    .pivot("q_rn", [1, 5])
    .agg(F.first("metric_value"))
    .withColumnRenamed("1", "eps_q0")
    .withColumnRenamed("5", "eps_q4")
    .withColumn(
        "qtr_eps_yoy_pct",
        F.when(
            (F.col("eps_q0").isNotNull()) & (F.col("eps_q4").isNotNull()) & (F.col("eps_q4") > 0),
            (F.col("eps_q0") / F.col("eps_q4") - 1) * 100
        ).otherwise(None)
    )
)

# Quarterly Revenue YoY
w_qtr_rev = Window.partitionBy("symbol").orderBy(F.to_date("period_end").desc())

qtr_rev_yoy = (
    fund_long
    .filter((F.col("period_type") == "Quarterly") & (F.col("metric_name") == "Total Revenue"))
    .withColumn("q_rn", F.row_number().over(w_qtr_rev))
    .filter(F.col("q_rn").isin(1, 5))
    .groupBy("symbol")
    .pivot("q_rn", [1, 5])
    .agg(F.first("metric_value"))
    .withColumnRenamed("1", "rev_q0")
    .withColumnRenamed("5", "rev_q4")
    .withColumn(
        "qtr_rev_yoy_pct",
        F.when(
            (F.col("rev_q0").isNotNull()) & (F.col("rev_q4").isNotNull()) & (F.col("rev_q4") > 0),
            (F.col("rev_q0") / F.col("rev_q4") - 1) * 100
        ).otherwise(None)
    )
)

# ------------------------------------------------------------
# 7. Join Everything
# ------------------------------------------------------------
market_elig = (
    market_snap
    .join(eligible, on="symbol", how="inner")
    .select(
        "symbol", "yahoo_symbol", "company_name", "sector", "industry",
        "market_cap", "shares_outstanding", "trailing_eps", "book_value"
    )
)

base = market_elig
base = base.join(ann_pivot, on="symbol", how="left")
base = base.join(ann_eps_pivot, on="symbol", how="left")
base = base.join(qtr_pivot_latest, on="symbol", how="left")
base = base.join(qtr_lag4, on="symbol", how="left")
base = base.join(ann_4y_agg, on="symbol", how="left")
base = base.join(qtr_eps_yoy, on="symbol", how="left")
base = base.join(qtr_rev_yoy, on="symbol", how="left")

# ------------------------------------------------------------
# 8. Compute All Ratios (Old + New)
# ------------------------------------------------------------
fund_features = (
    base
    # Capital Employed
    .withColumn(
        "capital_employed",
        F.coalesce(F.col("Total_Equity"), F.lit(0.0)) +
        F.coalesce(F.col("Total Debt"), F.lit(0.0)) -
        F.coalesce(F.col("Cash And Cash Equivalents"), F.lit(0.0))
    )
    # ROIC
    .withColumn(
        "roic",
        F.when(
            (F.col("capital_employed") > 0) & (F.col("EBIT").isNotNull()) & (F.col("EBIT") > 0),
            F.col("EBIT") / F.col("capital_employed")
        ).otherwise(F.lit(None).cast("double"))
    )
    # Enterprise Value
    .withColumn(
        "enterprise_value",
        F.col("market_cap") +
        F.coalesce(F.col("Total Debt"), F.lit(0.0)) -
        F.coalesce(F.col("Cash And Cash Equivalents"), F.lit(0.0))
    )
    # Earnings Yield
    .withColumn(
        "earnings_yield",
        F.when(
            (F.col("enterprise_value") > 0) & (F.col("EBIT").isNotNull()) & (F.col("EBIT") > 0),
            F.col("EBIT") / F.col("enterprise_value")
        ).otherwise(F.lit(None).cast("double"))
    )
    # Working Capital
    .withColumn(
        "working_capital",
        F.coalesce(
            F.col("Total Current Assets") - F.col("Total Current Liabilities"),
            F.col("capital_employed")
        )
    )
    # Current Ratio
    .withColumn(
        "current_ratio",
        F.when(
            (F.col("Total Current Liabilities").isNotNull()) & (F.col("Total Current Liabilities") > 0),
            F.col("Total Current Assets") / F.col("Total Current Liabilities")
        ).otherwise(F.lit(None).cast("double"))
    )
    # Debt to Equity
    .withColumn(
        "debt_to_equity",
        F.when(
            (F.col("Total_Equity").isNotNull()) & (F.col("Total_Equity") > 0) & (F.col("Total Debt").isNotNull()),
            F.col("Total Debt") / F.col("Total_Equity")
        ).otherwise(F.lit(None).cast("double"))
    )
    # OCF to NI
    .withColumn(
        "ocf_to_ni",
        F.when(
            (F.col("Net Income").isNotNull()) & (F.col("Net Income") > 0) & (F.col("Operating Cash Flow").isNotNull()),
            F.col("Operating Cash Flow") / F.col("Net Income")
        ).otherwise(F.lit(None).cast("double"))
    )
    # Quarterly EPS YoY (for scoring)
    .withColumn(
        "quarterly_eps_yoy",
        F.when(
            (F.col("Diluted_EPS_qtr").isNotNull()) & (F.col("Diluted_EPS_qtr_lag4").isNotNull()) & (F.col("Diluted_EPS_qtr_lag4") > 0),
            (F.col("Diluted_EPS_qtr") / F.col("Diluted_EPS_qtr_lag4")) - 1.0
        ).otherwise(F.lit(None).cast("double"))
    )
    # Quarterly Sales YoY (for scoring)
    .withColumn(
        "quarterly_sales_yoy",
        F.when(
            (F.col("Total_Revenue_qtr").isNotNull()) & (F.col("Total_Revenue_qtr_lag4").isNotNull()) & (F.col("Total_Revenue_qtr_lag4") > 0),
            (F.col("Total_Revenue_qtr") / F.col("Total_Revenue_qtr_lag4")) - 1.0
        ).otherwise(F.lit(None).cast("double"))
    )
    # 3-Year EPS CAGR (for scoring)
    .withColumn(
        "eps_cagr_3y",
        F.when(
            (F.col("Diluted_EPS_ann").isNotNull()) & (F.col("Diluted_EPS_ann_lag3").isNotNull()) &
            (F.col("Diluted_EPS_ann_lag3") > 0) & (F.col("Diluted_EPS_ann") > 0),
            F.pow(F.col("Diluted_EPS_ann") / F.col("Diluted_EPS_ann_lag3"), 1.0/3.0) - 1.0
        ).otherwise(F.lit(None).cast("double"))
    )
    # P/E
    .withColumn(
        "pe_proxy",
        F.when(
            (F.col("market_cap").isNotNull()) & (F.col("Net Income").isNotNull()) & (F.col("Net Income") > 0),
            F.col("market_cap") / F.col("Net Income")
        ).otherwise(F.lit(None).cast("double"))
    )
    # P/B
    .withColumn(
        "pb",
        F.when(
            (F.col("market_cap").isNotNull()) & (F.col("Total_Equity").isNotNull()) & (F.col("Total_Equity") > 0),
            F.col("market_cap") / F.col("Total_Equity")
        ).otherwise(F.lit(None).cast("double"))
    )
    # PEG
    .withColumn(
        "peg",
        F.when(
            (F.col("pe_proxy").isNotNull()) & (F.col("eps_cagr_3y").isNotNull()) & (F.col("eps_cagr_3y") > 0),
            F.col("pe_proxy") / (F.col("eps_cagr_3y") * 100.0)
        ).otherwise(F.lit(None).cast("double"))
    )
)

# ------------------------------------------------------------
# 9. Validity Flags (same as old)
# ------------------------------------------------------------
fund_final = (
    fund_features
    .withColumn(
        "financial_fresh_flag",
        F.when(
            F.col("period_end_dt").isNotNull(),
            F.datediff(F.current_date(), F.col("period_end_dt")) < 730
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "roic_valid",
        (F.col("capital_employed").isNotNull()) & (F.col("capital_employed") > 0) &
        (F.col("EBIT").isNotNull()) & (F.col("EBIT") > 0)
    )
    .withColumn(
        "ey_valid",
        (F.col("enterprise_value").isNotNull()) & (F.col("enterprise_value") > 0) &
        (F.col("EBIT").isNotNull()) & (F.col("EBIT") > 0)
    )
    .withColumn(
        "fundamentals_valid",
        (F.col("market_cap").isNotNull()) &
        (F.col("Total Debt").isNotNull()) &
        (F.col("Total_Equity").isNotNull()) &
        (F.col("Operating Cash Flow").isNotNull()) &
        (F.col("Net Income").isNotNull()) &
        (F.col("EBIT").isNotNull())
    )
    .withColumn("calculation_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("run_id", F.lit(RUN_ID))
)

# ------------------------------------------------------------
# 10. Select ALL Columns (Old + New) and Write
# ------------------------------------------------------------
silver_fund = (
    fund_final
    .dropDuplicates(["symbol"])
    .select(
        # Identity
        "symbol", "yahoo_symbol", "company_name", "sector", "industry",
        "period_end_dt", "period_end_dt_q",
        "market_cap", "shares_outstanding",
        # Balance Sheet (latest)
        "Total Current Assets", "Total Current Liabilities",
        "working_capital", "Net PPE", "capital_employed",
        "Total Debt", "Cash And Cash Equivalents",
        "enterprise_value", "Total_Equity",
        # P&L (latest)
        "EBIT", "Total Revenue", "Net Income", "Operating Cash Flow",
        # EPS lags (for CAGR)
        "Diluted_EPS_ann", "Diluted_EPS_ann_lag1",
        "Diluted_EPS_ann_lag2", "Diluted_EPS_ann_lag3",
        # Quarterly (latest + lag4)
        "Diluted_EPS_qtr", "Diluted_EPS_qtr_lag4",
        "Total_Revenue_qtr", "Total_Revenue_qtr_lag4",
        # Ratios (for scoring)
        "roic", "earnings_yield", "current_ratio",
        "debt_to_equity", "ocf_to_ni",
        "quarterly_eps_yoy", "quarterly_sales_yoy",
        "eps_cagr_3y", "pe_proxy", "pb", "peg",
        # Flags
        "financial_fresh_flag", "roic_valid", "ey_valid", "fundamentals_valid",
        # 4-Year Annual Metrics
        "eps_y0", "eps_y1", "eps_y2", "eps_y3",
        "eps_cagr_4y",
        "debt_y0", "debt_y3", "is_deleveraging",
        "rev_y0", "rev_y3", "rev_cagr_4y",
        "ocf_y0", "ocf_y3", "ocf_cagr_4y",
        "cum_4yr_ocf", "cum_4yr_ni", "cum_4yr_ocf_to_ni",
        "positive_earnings_4y_count",
        # Quarterly YoY (1-year)
        "eps_q0", "eps_q4", "qtr_eps_yoy_pct",
        "rev_q0", "rev_q4", "qtr_rev_yoy_pct",
        # Audit
        "calculation_timestamp", "run_id"
    )
    .withColumnRenamed("period_end_dt", "annual_period_end")
    .withColumnRenamed("period_end_dt_q", "quarterly_period_end")
    .withColumnRenamed("Total Current Assets", "total_current_assets")
    .withColumnRenamed("Total Current Liabilities", "total_current_liabilities")
    .withColumnRenamed("Net PPE", "net_ppe")
    .withColumnRenamed("Total Debt", "total_debt")
    .withColumnRenamed("Cash And Cash Equivalents", "cash_and_equivalents")
    .withColumnRenamed("Total_Equity", "total_equity")
    .withColumnRenamed("EBIT", "ebit")
    .withColumnRenamed("Total Revenue", "total_revenue_ann")
    .withColumnRenamed("Net Income", "net_income_ann")
    .withColumnRenamed("Operating Cash Flow", "operating_cash_flow_ann")
    .withColumnRenamed("Diluted_EPS_ann", "diluted_eps_ann")
    .withColumnRenamed("Diluted_EPS_ann_lag1", "diluted_eps_ann_lag1")
    .withColumnRenamed("Diluted_EPS_ann_lag2", "diluted_eps_ann_lag2")
    .withColumnRenamed("Diluted_EPS_ann_lag3", "diluted_eps_ann_lag3")
    .withColumnRenamed("Diluted_EPS_qtr", "diluted_eps_qtr")
    .withColumnRenamed("Diluted_EPS_qtr_lag4", "diluted_eps_qtr_lag4")
    .withColumnRenamed("Total_Revenue_qtr", "total_revenue_qtr")
    .withColumnRenamed("Total_Revenue_qtr_lag4", "total_revenue_qtr_lag4")
)

# ------------------------------------------------------------
# 11. Sector-Relative Z-Scores
#     Placed after the final `silver_fund` selection/rename (step 10),
#     since it needs the already-renamed pe_proxy/pb/earnings_yield/roic
#     columns plus `sector`, all of which only exist after that select.
#     NOTE: sectors with a single stock (or a stock as the only non-null
#     value for a given ratio) get NULL here, not 0 — stddev over one row
#     is undefined in Spark, and F.stddev of one non-null value is NULL,
#     so 0/NULL -> NULL rather than a misleading "average" z-score of 0.
# ------------------------------------------------------------
w_sector = Window.partitionBy("sector")

silver_fund = (
    silver_fund
    .withColumn(
        "pe_sector_zscore",
        (F.col("pe_proxy") - F.avg("pe_proxy").over(w_sector)) /
        F.stddev("pe_proxy").over(w_sector)
    )
    .withColumn(
        "pb_sector_zscore",
        (F.col("pb") - F.avg("pb").over(w_sector)) /
        F.stddev("pb").over(w_sector)
    )
    .withColumn(
        "ey_sector_zscore",
        (F.col("earnings_yield") - F.avg("earnings_yield").over(w_sector)) /
        F.stddev("earnings_yield").over(w_sector)
    )
    .withColumn(
        "roic_sector_zscore",
        (F.col("roic") - F.avg("roic").over(w_sector)) /
        F.stddev("roic").over(w_sector)
    )
)

(
    silver_fund.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_fundamental_features")
)

print(f"✅ SUCCESS: {silver_fund.count()} stocks with 4-year metrics + sector z-scores")
print(f"   Columns: {len(silver_fund.columns)}")

StatementMeta(, e3309cd0-53f1-4aa4-a17e-7468f501b3b1, 3, Finished, Available, Finished, False)

RUN_ID: 5815daa6-944c-4fce-b1ca-9f6ea6737609
RUN_TIMESTAMP: 2026-09-20 03:48:05
Eligible stocks: 398
✅ SUCCESS: 398 stocks with 4-year metrics + sector z-scores
   Columns: 75


In [ ]:
display(
    spark.sql("""
        SELECT
            symbol,
            annual_period_end,
            quarterly_period_end,
            market_cap,
            roic,
            earnings_yield,
            current_ratio,
            debt_to_equity,
            ocf_to_ni,
            pe_proxy,
            pb,
            peg,
            eps_cagr_3y
        FROM silver_fundamental_features
        ORDER BY symbol
        LIMIT 15
    """)
)

StatementMeta(, d99298a4-88df-4115-accd-f953c82a46db, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.sql("""
        SELECT
            symbol,
            quarterly_eps_yoy,
            quarterly_sales_yoy,
            eps_cagr_3y,
            diluted_eps_ann,
            diluted_eps_ann_lag1,
            diluted_eps_ann_lag2,
            diluted_eps_ann_lag3
        FROM silver_fundamental_features
        ORDER BY eps_cagr_3y DESC
        LIMIT 15
    """)
)

StatementMeta(, d99298a4-88df-4115-accd-f953c82a46db, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.sql("""
        SELECT
            roic_valid,
            ey_valid,
            fundamentals_valid,
            financial_fresh_flag,
            COUNT(*) AS stock_count
        FROM silver_fundamental_features
        GROUP BY
            roic_valid,
            ey_valid,
            fundamentals_valid,
            financial_fresh_flag
        ORDER BY stock_count DESC
    """)
)

StatementMeta(, d99298a4-88df-4115-accd-f953c82a46db, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.sql("""
        SELECT
            symbol,
            annual_period_end,
            DATEDIFF(current_date(), annual_period_end) AS days_old,
            financial_fresh_flag
        FROM silver_fundamental_features
        ORDER BY days_old DESC
        LIMIT 15
    """)
)

StatementMeta(, d99298a4-88df-4115-accd-f953c82a46db, -1, Cancelled, , Cancelled, True)

In [ ]:
display(
    spark.sql("""
        SELECT
            symbol,
            net_ppe,
            capital_employed,
            roic_valid
        FROM silver_fundamental_features
        WHERE net_ppe IS NULL
        ORDER BY symbol
        LIMIT 20
    """)
)

StatementMeta(, d99298a4-88df-4115-accd-f953c82a46db, -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

fund = spark.read.table("silver_fundamental_features")
print(f"Before: {fund.count()}")

w = Window.partitionBy("symbol").orderBy(F.col("calculation_timestamp").desc())
clean = (
    fund
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
print(f"After: {clean.count()}")

(
    clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_fundamental_features")
)
print("Dedup complete.")

StatementMeta(, d99298a4-88df-4115-accd-f953c82a46db, -1, Cancelled, , Cancelled, True)